In [11]:
# =============================================================================
# FINE-TUNING WHISPER ON YOUR DOMAIN
# =============================================================================
#
# Course map:
#   Phase 04 → Transfer learning & fine-tuning (general)
#   Phase 06 → Whisper architecture & fine-tuning
#   Phase 11 → LoRA & QLoRA (train few extra weights, keep base frozen)
#
# Today's deliverable (target):
#   LoRA-tuned Whisper that hears SALES jargon better
#   (WER on terms like "SOC 2", "$299", "Professional" ~25% → <5%)
#   Drop-in replacement for the ASR step in your live earpiece loop.
#
# ---------------------------------------------------------------------------
# THE PROBLEM (easy picture)
# ---------------------------------------------------------------------------
# Stock Whisper is great on general English. On a sales call it still mishears:
#   "SOC 2" → "sock too"     "$299" → "two ninety nine" messy
#   "Professional plan" → "professional planned"
#
# Your earpiece Correction-GPT then starts from BAD text → worse coaching.
#
# Fine-tuning = show Whisper many (audio → correct transcript) examples from
# YOUR domain so those words become easy.
#
#   Generic Whisper          Domain-tuned Whisper
#   "sock too compliant"  →  "SOC 2 compliant"
#
# ---------------------------------------------------------------------------
# WHAT IS FINE-TUNING? (deep but easy)
# ---------------------------------------------------------------------------
# Fine-tuning = take a model that ALREADY learned a general skill, then keep
# training it a bit more on YOUR smaller, specialized dataset so it gets
# better at your job.
#
# Picture:
#   1) PRETRAIN (already done by OpenAI for Whisper)
#        Millions of hours of speech → model learns English sounds & spelling.
#        Like finishing school.
#
#   2) FINE-TUNE (what YOU do in this notebook)
#        Show pairs:  [sales audio] → "The Professional plan costs $299…"
#        Update weights (or LoRA adapters) so mistakes on sales words shrink.
#        Like an internship: same brain, new workplace vocabulary.
#
#   3) INFERENCE (your live agent)
#        Use the tuned model to transcribe new calls. No labels needed then.
#
# What changes under the hood?
#   Training still minimizes a loss (Whisper: predict next transcript tokens
#   given the audio). Difference vs "training from scratch":
#     - start from strong pretrained weights (not random)
#     - fewer steps / smaller LR usually
#     - dataset is domain-specific and much smaller
#
# Fine-tuning vs related ideas:
#   Transfer learning  — broad name for "reuse pretrained knowledge"
#   Fine-tuning        — the common transfer method: continue gradient updates
#                        on the pretrained net (full or LoRA)
#   Prompting / RAG    — change INPUTS, not weights (no training)
#   SFT (week2)        — fine-tuning a text LLM on instruction→answer flashcards
#                        (same IDEA as here; different modality: text vs speech)
#
# Sticky one-liner:
#   Fine-tuning = specialized practice for a pretrained model.
#
# ---------------------------------------------------------------------------
# FULL FINE-TUNE vs LoRA vs QLoRA (slow walkthrough)
# ---------------------------------------------------------------------------
# Think of Whisper as a HUGE binder of knobs (millions of weights).
# Fine-tuning means: turn some knobs so sales audio is recognized better.
#
# --- 1) FULL FINE-TUNE (Full FT) ---
#   What: unlock EVERY knob and train them all on your sales clips.
#   Pros: maximum flexibility; can change behavior a lot.
#   Cons:
#     - Needs a strong GPU / lots of memory
#     - With only 50–100 clips, the model can MEMORIZE those clips
#       and forget general English (catastrophic forgetting / overfit)
#     - Checkpoint is HUGE (you save the whole model again)
#
#   Picture: repainting the entire house to match one room's style.
#
# --- 2) LoRA (Low-Rank Adaptation) ---
#   What: FREEZE the original Whisper knobs (leave school knowledge intact).
#         Add tiny extra matrices (adapters) next to certain layers
#         (often attention projections). Train ONLY those tiny adapters.
#
#   Math intuition (no pain):
#     A big weight update would be a giant matrix ΔW.
#     LoRA says: approximate ΔW ≈ A × B where A,B are skinny/small ("low rank").
#     Far fewer numbers to learn → cheaper + less overfit on tiny data.
#
#   Pros:
#     - Train ~1% (or less) of parameters
#     - Small adapter file (MBs), base Whisper stays shared
#     - Swap adapters: sales_lora.pt vs medical_lora.pt without copying base
#   Cons:
#     - Slightly less flexible than full FT if you need a huge behavior change
#
#   Picture: leave the house painted; stick removable STYLE DECALS on doors.
#            Want a new domain? Peel decals, stick different ones.
#
#       ┌─────────────────────────────┐
#       │  Frozen Whisper weights     │  ← not updated
#       │    + LoRA adapters (train)  │  ← only these learn sales words
#       └─────────────────────────────┘
#
# --- 3) QLoRA (Quantized LoRA) ---
#   What: same LoRA idea, but the FROZEN base is stored in 4-bit (compressed
#         numbers) to save GPU RAM. Adapters still train in higher precision.
#
#   Quantization (simple):
#     Store weights with fewer bits (less detail) ≈ zip file for numbers.
#     Model is a bit "blurrier" but much smaller in memory.
#
#   Pros: fine-tune bigger Whisper variants on smaller GPUs / laptops
#   Cons: setup is pickier (bitsandbytes, GPU drivers); tiny quality tradeoff
#
#   Picture: keep the house as a compressed photo album (4-bit) + train
#            the same small decals (LoRA) on top.
#
# --- Which should YOU use for this sales Whisper project? ---
#   Tiny demo dataset (tens of clips) → prefer LoRA (or QLoRA if VRAM tight)
#   Full FT → only if you have lots of labeled audio + serious GPU
#
# Sticky cheat sheet:
#   Full FT = retrain whole brain
#   LoRA    = freeze brain, train small stickers
#   QLoRA   = freeze a COMPRESSED brain, train small stickers
#
# ---------------------------------------------------------------------------
# ABBREVIATIONS
# ---------------------------------------------------------------------------
#   ASR  = Automatic Speech Recognition — speech → text (Whisper)
#   WER  = Word Error Rate — % of words wrong vs a gold transcript
#          (insertions + deletions + substitutions) / N_ref_words
#   LoRA = Low-Rank Adaptation — parameter-efficient fine-tuning method
#   TTS  = Text-To-Speech — can SYNTHESIZE training audio from scripts
#          (good for bootstrapping; real calls are better later)
#
# ---------------------------------------------------------------------------
# DATA YOU NEED
# ---------------------------------------------------------------------------
#   Pairs:  audio.wav  +  exact text that was said
#   Start:  50–100 sales lines (TTS or read aloud)
#   Better: real anonymized call snippets with human transcripts
#
# Pipeline in this notebook:
#   1) Build (audio, text) dataset
#   2) Load whisper-tiny.en + processor
#   3) (Later cells) LoRA train → evaluate WER → plug into live agent ASR
#
print("Whisper domain fine-tuning map loaded.")


Whisper domain fine-tuning map loaded.


In [12]:
# =============================================================================
# PREPARE SALES AUDIO + TRANSCRIPTS — (audio path, text) pairs for Whisper
# =============================================================================
#
# Goal of this cell:
#   Build a HuggingFace Dataset with columns like:
#     audio  → waveform @ 16 kHz (what Whisper hears)
#     text   → gold transcript (what Whisper should type)
#
# Production: real calls. Here: create a tiny sales_audio/ folder automatically
# if you don't have one yet (placeholder tones + real sales sentences).
# Replace with TTS (pyttsx3/Coqui) or mic recordings when you go serious.
#
# Terms:
#   WhisperProcessor — turns audio↔features and text↔token ids for Whisper
#   WhisperForConditionalGeneration — the seq2seq ASR model
#   sampling_rate 16000 — Whisper English checkpoints expect 16 kHz mono
#   metadata.csv — simple table: file name + transcript
#

import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import wave

from datasets import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

AUDIO_DIR = Path("sales_audio")
META_CSV = AUDIO_DIR / "metadata.csv"
SAMPLE_RATE = 16000

# Domain lines you CARE about (pricing, compliance, plan names)
SALES_LINES = [
    ("call1.wav", "The Professional plan costs $299 per month."),
    ("call2.wav", "We are SOC 2 Type II compliant."),
    ("call3.wav", "Starter is $99 and includes 24/7 support."),
    ("call4.wav", "Enterprise customers get phone support with a one hour SLA."),
    ("call5.wav", "Rate limits are 5000 requests per minute for Professional."),
]


def _write_placeholder_wav(path: Path, seconds: float = 1.0, freq: float = 220.0):
    """Write a short mono 16-bit WAV (tone). Structure demo only — not real speech."""
    t = np.linspace(0, seconds, int(SAMPLE_RATE * seconds), endpoint=False)
    # Quiet tone so the file is valid audio; swap for TTS speech later
    audio = (0.1 * np.sin(2 * np.pi * freq * t) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SAMPLE_RATE)
        wf.writeframes(audio.tobytes())


# Create folder + CSV + wavs if missing
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
if not META_CSV.exists():
    rows = []
    for i, (fname, text) in enumerate(SALES_LINES):
        wav_path = AUDIO_DIR / fname
        if not wav_path.exists():
            _write_placeholder_wav(wav_path, seconds=1.0 + 0.1 * i, freq=200 + 20 * i)
        rows.append({"file": fname, "text": text})
    pd.DataFrame(rows).to_csv(META_CSV, index=False)
    print(f"Created {META_CSV} with {len(rows)} placeholder clips.")
else:
    print(f"Using existing {META_CSV}")

df = pd.read_csv(META_CSV)


def _load_wav(path: Path) -> dict:
    """Load mono float32 waveform with stdlib wave — no torchcodec needed.

    datasets>=4 Audio() decoding requires the torchcodec package and a
    path→Audio cast that can break with pandas large_string columns.
    We already write 16 kHz WAVs above, so decode them ourselves.
    """
    with wave.open(str(path), "rb") as wf:
        assert wf.getframerate() == SAMPLE_RATE, f"expected {SAMPLE_RATE} Hz, got {wf.getframerate()}"
        n_ch = wf.getnchannels()
        raw = wf.readframes(wf.getnframes())
        arr = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
        if n_ch > 1:
            arr = arr.reshape(-1, n_ch).mean(axis=1)
    return {"array": arr, "sampling_rate": SAMPLE_RATE}


rows = []
for _, row in df.iterrows():
    rows.append(
        {
            "file": row["file"],
            "text": row["text"],
            "audio": _load_wav(AUDIO_DIR / row["file"]),
        }
    )

dataset = Dataset.from_list(rows)

print("Sample row keys:", dataset[0].keys())
print("text:", dataset[0]["text"])
print("audio sampling_rate:", dataset[0]["audio"]["sampling_rate"])
print("audio array length:", len(dataset[0]["audio"]["array"]))

# Base English Whisper (tiny = fast for learning; upgrade to small/medium later)
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")

print("Loaded openai/whisper-tiny.en | trainable params:",
      sum(p.numel() for p in model.parameters()))
print("Next: freeze base + attach LoRA, then train on these (audio, text) pairs.")
print("NOTE: placeholder WAVs teach the pipeline; for real WER gains use TTS/real speech.")


Using existing sales_audio/metadata.csv
Sample row keys: dict_keys(['file', 'text', 'audio'])
text: The Professional plan costs $299 per month.
audio sampling_rate: 16000
audio array length: 16000


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 13598.04it/s]


Loaded openai/whisper-tiny.en | trainable params: 37760256
Next: freeze base + attach LoRA, then train on these (audio, text) pairs.
NOTE: placeholder WAVs teach the pipeline; for real WER gains use TTS/real speech.


In [13]:
# =============================================================================
# PREPROCESS DATA — turn (raw audio, text) into what Whisper trains on
# =============================================================================
#
# Sticky one-liner:
#   Preprocessing = convert HUMAN-friendly data → MODEL-friendly tensors
#   BEFORE the training loop. Same idea as tokenization for LLMs, but for
#   speech you also convert waveforms into spectrogram features.
#
# ---------------------------------------------------------------------------
# WHERE THIS SITS IN THE PIPELINE
# ---------------------------------------------------------------------------
#   RAW WORLD                 PREPROCESS (this cell)              TRAIN
#   -----------               ----------------------              -----
#   call1.wav  ─────────┐
#   "Professional…"     ├──→  input_features (log-Mel)  ──┐
#                       │     labels (token ids)          ├──→ loss / LoRA
#   (from previous cell)┘                                 │
#
# If you skip preprocess and feed raw float samples + English strings into
# Whisper, the model cannot learn — it only speaks "tensor language".
#
# ---------------------------------------------------------------------------
# WHAT IS PREPROCESSING? (deep but easy)
# ---------------------------------------------------------------------------
# Three jobs, always:
#   1) CLEAN / NORMALIZE  — make inputs comparable (rate, mono, scale, trim)
#   2) FEATURIZE          — turn signal into the representation the model expects
#   3) ALIGN LABELS       — encode targets the same way the model will decode
#
# For Whisper ASR specifically:
#   audio  → log-Mel spectrogram  →  input_features  shape (80, 3000)
#   text   → BPE token ids        →  labels          list of ints
#
# Picture (audio path):
#   waveform samples
#        │  STFT (short windows of sound)
#        ▼
#   frequency × time energy map
#        │  mel filterbank (human-ear frequency spacing)
#        ▼
#   80 mel bands × time frames
#        │  log + Whisper's per-feature normalize
#        ▼
#   input_features  ← THIS is what the encoder "hears"
#
# Picture (text path):
#   "The Professional plan costs $299 per month."
#        │  WhisperTokenizer (BPE + special tokens)
#        ▼
#   [50257, 50362, 464, 18612, ...]  ← decoder learning targets
#
# ---------------------------------------------------------------------------
# TYPES OF PREPROCESSING (map of the industry)
# ---------------------------------------------------------------------------
# A) DATA-QUALITY / CLEANING (often offline, before Dataset)
#    - silence trim, denoise, gain normalize, drop corrupt files
#    - language / PII filtering, transcript QA
#    Industry: common in ASR data pipelines (Kaldi / NVIDIA NeMo / SpeechBrain)
#
# B) SIGNAL / ACOUSTIC NORMALIZATION
#    - resample to model rate (Whisper: 16 kHz mono)
#    - peak / loudness normalize (e.g. target LUFS in production podcasts)
#    - channel mix (stereo → mono)
#    Industry standard for English Whisper: 16_000 Hz, mono, float in [-1, 1]
#
# C) FEATURE EXTRACTION (what THIS cell focuses on for audio)
#    Classic ASR eras used different "ears":
#      MFCC          — Mel-Frequency Cepstral Coefficients (GMM-HMM / early DNN)
#      Filterbank / Mel spectrogram — most modern neural ASR
#      Raw waveform  — some end-to-end models (Wav2Vec 2.0 learns its own front-end)
#      log-Mel       — Whisper / many Transformer ASR models (OpenAI standard)
#    Whisper fixed recipe (industry de-facto for this family):
#      - 25 ms window (n_fft=400 @ 16 kHz), hop 10 ms (160 samples)
#      - 80 mel bins
#      - pad / truncate to 30 seconds → 3000 time frames
#      → tensor shape (80, 3000) per clip
#
# D) TEXT / LABEL PREPROCESSING
#    - normalize transcripts (numbers, casing, punctuation policy)
#    - tokenize with the SAME tokenizer the checkpoint was trained with
#    - special tokens: Whisper uses start/transcript markers around the text
#    Industry: NEVER mix tokenizers across models; mismatch = silent garbage.
#
# E) AUGMENTATION (train-time preprocess; optional later cell)
#    - SpecAugment (mask time/freq on the Mel), speed perturb, noise mix
#    Industry: SpecAugment is a standard for Transformer ASR; helps tiny datasets.
#
# F) BATCHING / COLLATION (DataLoader preprocess)
#    - pad variable-length labels to a batch max
#    - mask padding with -100 so CrossEntropy ignores it (HF Trainer convention)
#    Not in this cell yet — comes when you wire Trainer / custom collator.
#
# G) LLM-SIDE PREPROCESS (for comparison — week2 SFT)
#    - chat template + BPE tokenize + label mask on prompt tokens
#    Same IDEA as here: strings → ids. Different MODALITY on the input side.
#
# ---------------------------------------------------------------------------
# INDUSTRY STANDARDS (what "good" looks like)
# ---------------------------------------------------------------------------
# Speech ASR:
#   ✓ Match the pretrained model's front-end exactly (rate, mel, hop, pad)
#   ✓ Train / eval transcript normalization policy must match how you score WER
#   ✓ Prefer storing paths + decode on the fly for big corpora; tiny demo = OK
#      to materialize features in RAM (what we do below)
#   ✓ Document special-token handling (Whisper: decoder prompt tokens in labels)
#   ✗ Don't invent your own Mel and expect a pretrained Whisper to transfer
#
# Vision / multimodal (same philosophy elsewhere):
#   ImageNet-style mean/std, fixed resize/crop — always match the checkpoint.
#
# Text LLMs:
#   Chat template + tokenizer from THAT model card — same rule as WhisperTokenizer.
#
# Rule of thumb used in labs & production:
#   "Preprocess like the original training run, or fine-tuning is fighting you."
#
# ---------------------------------------------------------------------------
# ABBREVIATIONS
# ---------------------------------------------------------------------------
#   STFT     = Short-Time Fourier Transform — windowed frequency analysis
#   Mel      = Mel scale — frequency axis warped like human hearing
#   log-Mel  = log of mel filterbank energies (Whisper input)
#   BPE      = Byte-Pair Encoding — tokenizer Whisper uses for text
#   WER      = Word Error Rate — how we judge ASR quality later
#   LUFS     = loudness units (broadcast / podcast loudness standard)
#   SpecAug  = SpecAugment — mask patches on spectrogram as augmentation
#
# ---------------------------------------------------------------------------
# THIS CELL'S CODE (what each line does)
# ---------------------------------------------------------------------------
#   processor = WhisperProcessor...
#       Loads BOTH:
#         feature_extractor  → audio → input_features
#         tokenizer          → text  → labels
#       Always load the SAME checkpoint name as the model ("tiny.en").
#
#   prepare_dataset(batch):
#       audio["array"]  → feature_extractor → batch["input_features"]
#       batch["text"]   → tokenizer         → batch["labels"]
#
#   dataset.map(..., remove_columns=...)
#       Drops raw audio/text columns; keeps only tensors the trainer needs.
#
#   set_format("torch", ...)
#       So next cells get torch.Tensors instead of Python lists.
#
# ---------------------------------------------------------------------------
# EXPECTED OUTPUT (shapes to remember)
# ---------------------------------------------------------------------------
#   input_features[0]  → shape (80, 3000)
#       80  = mel bins (frequency "piano keys")
#       3000 = time frames for a 30s Whisper chunk (short clips are zero-padded)
#   labels[0]          → e.g. ~10–40 ints for our short sales sentences
#       Starts with Whisper special ids, then BPE pieces of the transcript
#   After decode(labels, skip_special_tokens=True) → original English text
#

# Reuse processor if the previous cell already loaded it; otherwise load here.
try:
    processor
except NameError:
    processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")


def prepare_dataset(batch):
    """One example → Whisper training tensors.

    Input batch (from previous cell):
      batch["audio"]["array"]          float32 waveform @ 16 kHz
      batch["audio"]["sampling_rate"]  16000
      batch["text"]                    gold transcript string

    Output batch:
      batch["input_features"]  log-Mel, length-80×3000 (list/np before set_format)
      batch["labels"]          token ids the decoder should emit
    """
    audio = batch["audio"]

    # --- AUDIO → log-Mel features (encoder input) ---
    # WhisperFeatureExtractor:
    #   1) optional resample (we already are at 16 kHz)
    #   2) STFT → mel filterbank → log
    #   3) pad/trim to 30s → (80, 3000)
    feats = processor.feature_extractor(
        audio["array"],
        sampling_rate=16000,  # MUST match Whisper English checkpoints
    )
    batch["input_features"] = feats.input_features[0]

    # --- TEXT → token ids (decoder targets) ---
    # Same tokenizer Whisper was pretrained with (BPE + special tokens).
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


# Map over the whole tiny sales set (5 clips). For 10k+ hours you'd:
#   - decode audio lazily, cache features on disk, or use streaming datasets.
dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)
dataset.set_format(type="torch", columns=["input_features", "labels"])

# ----- walk the OUTPUT so you can "see" preprocess -----
ex = dataset[0]
feat = ex["input_features"]
labs = ex["labels"]

print("columns after preprocess:", dataset.column_names)
print("input_features shape:", tuple(feat.shape), " dtype:", feat.dtype)
print("  → 80 mel bins × 3000 time frames (Whisper 30s canvas)")
print("labels (token ids):", labs.tolist() if hasattr(labs, "tolist") else labs)
print("labels length:", len(labs))
print(
    "labels → text:",
    processor.tokenizer.decode(labs, skip_special_tokens=True),
)
print("Ready for training: each row is (encoder Mel, decoder token targets).")


Map: 100%|██████████| 5/5 [00:00<00:00, 312.27 examples/s]

columns after preprocess: ['input_features', 'labels']
input_features shape: (80, 3000)  dtype: torch.float32
  → 80 mel bins × 3000 time frames (Whisper 30s canvas)
labels (token ids): [50257, 50362, 464, 18612, 1410, 3484, 720, 22579, 583, 1227, 13, 50256]
labels length: 12
labels → text: The Professional plan costs $299 per month.
Ready for training: each row is (encoder Mel, decoder token targets).


In [15]:
# Apply LoRA to Whisper

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Whisper uses these in attention layers
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Should print < 1% of total parameters.

Loading weights: 100%|██████████| 167/167 [00:00<00:00, 14083.34it/s]


trainable params: 147,456 || all params: 37,907,712 || trainable%: 0.3890


In [ ]:
# Training Setup (Seq2SeqTrainer)
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-sales",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=50,
    num_train_epochs=3,
    logging_steps=10,
    evaluation_strategy="no",
    save_strategy="epoch",
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=processor.feature_extractor,
    # data_collator=... (we need a data collator for seq2seq)
)

# Data collator for Whisper:

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)
trainer.data_collator = data_collator

TypeError: Seq2SeqTrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [19]:
# Train and Save
trainer.train()
model.save_pretrained("./whisper-sales-lora")
processor.save_pretrained("./whisper-sales-lora")

NameError: name 'trainer' is not defined

In [20]:
# Evaluate Word Error Rate on Jargon
import evaluate
wer_metric = evaluate.load("wer")

def evaluate_on_test(test_samples):
    predictions = []
    references = []
    for sample in test_samples:
        audio = sample["audio"]["array"]
        input_feats = processor(audio, return_tensors="pt").input_features
        generated_ids = model.generate(input_feats)
        pred_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        predictions.append(pred_text)
        references.append(sample["text"])
    wer = wer_metric.compute(predictions=predictions, references=references)
    print(f"WER: {wer:.4f}")
    return wer

# Create a small test set with domain-specific phrases
test_data = [
    {"audio": "call_test1.wav", "text": "We are SOC 2 Type II compliant."},
    {"audio": "call_test2.wav", "text": "The Professional plan costs $299 per month."},
    # ...
]
evaluate_on_test(test_data)

# Expected: With fine‑tuning, WER on these sentences drops dramatically compared to base Whisper.

ModuleNotFoundError: No module named 'evaluate'

In [ ]:
# Integrate into the Live Assistant

# Replace the previous ASR class:
class FineTunedASR:
    def __init__(self, model_path="./whisper-sales-lora"):
        self.processor = WhisperProcessor.from_pretrained(model_path)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_path)
        self.model.eval()

    def transcribe(self, audio_bytes):
        # Save temp and load
        with open("temp.wav", "wb") as f:
            f.write(audio_bytes)
        audio, sr = torchaudio.load("temp.wav")
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(sr, 16000)
            audio = resampler(audio)
        input_feats = self.processor(audio.squeeze().numpy(), return_tensors="pt").input_features
        generated_ids = self.model.generate(input_feats)
        return self.processor.decode(generated_ids[0], skip_special_tokens=True)

# Now your assistant transcribes "SOC 2" correctly, never mishears "Professional" as "professionally," and knows 
# the difference between "$99" and "$299."